# Final GCN-3 Ensemble: Confidence and Prediction Deviations

This notebook runs the confirmed best overall configuration: a five-member plain GCN-3 ensemble with weighted edges, richer node features, graph-level features, target normalization, Adam, and the established two-phase schedule.

For every validation, test, and external prediction sample it saves the ensemble mean, member standard deviation, standard error, range, 80/90/95% Student-t confidence intervals for the mean, and every member's deviation from the ensemble mean. It also reports empirical interval coverage and the correlation between ensemble spread and actual error.

Important: these confidence intervals quantify disagreement across initialization seeds. They are not guaranteed predictive intervals for true stiffness, particularly under distribution shift.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'
if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone skipped outside Colab.')

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_uncertainty'
RESUMABLE_RUN_NAME = 'run_final_ensemble_uncertainty_v1'
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODELS_TO_GITHUB = False
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_BRANCH = 'main'
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/gcn3_ensemble_uncertainty'

repo_root = Path(REPO_DIR).resolve() if IN_COLAB else Path.cwd().resolve()
pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
gnn_root = pipeline_root / 'gnn_prototype'
optimization_dir = gnn_root / 'GCN_Optimization'
required_files = (
    'colab_gnn_stiffness_prototype.py',
    'architecture_comparison_runner.py',
    'ensemble_runner.py',
    'ensemble_uncertainty_runner.py',
)
missing = [name for name in required_files if not (optimization_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing ensemble modules under {optimization_dir}: {missing}')
os.chdir(pipeline_root)
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')
if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'
if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/gcn3_ensemble_uncertainty') if IN_COLAB else gnn_root / 'outputs' / 'gcn3_ensemble_uncertainty'
output_root.mkdir(parents=True, exist_ok=True)
git_output_root = repo_root / GIT_RESULTS_SUBDIR
if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email: ').strip()
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')

In [ ]:
import pandas as pd
import shutil
import subprocess
import torch
from IPython.display import Image, display
from ensemble_uncertainty_runner import EnsembleUncertaintyConfig, run_ensemble_uncertainty_experiment

In [ ]:
MEMBER_SEEDS = (11, 42, 73, 101, 202)
SPLIT_SEED = 42
CONFIDENCE_LEVELS = (0.80, 0.90, 0.95)
HIDDEN_DIM = 24
DROPOUT = 0.10
LR_PHASE1 = 0.003
LR_PHASE2 = 0.0005
EPOCHS_PHASE1 = 200
EPOCHS_PHASE2 = 700
PATIENCE = 999
WEIGHT_DECAY = 1e-5
CHECKPOINT_INTERVAL = 50
RESUME = True
REQUIRE_GPU = True

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Select Runtime > Change runtime type > T4 GPU.')
config = EnsembleUncertaintyConfig(
    member_seeds=MEMBER_SEEDS,
    split_seed=SPLIT_SEED,
    confidence_levels=CONFIDENCE_LEVELS,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    lr_phase1=LR_PHASE1,
    lr_phase2=LR_PHASE2,
    epochs_phase1=EPOCHS_PHASE1,
    epochs_phase2=EPOCHS_PHASE2,
    patience=PATIENCE,
    weight_decay=WEIGHT_DECAY,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    resume=RESUME,
)
run_dir = output_root / RESUMABLE_RUN_NAME
print(config)
print(f'Resumable run directory: {run_dir}')

In [ ]:
result = run_ensemble_uncertainty_experiment(
    config,
    train_root=train_root,
    predict_root=predict_root,
    run_dir=run_dir,
)
output_dir = result['output_dir']
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')
print('Ensemble performance:')
display(result['ensemble_summary'])
print('Confidence and coverage summary:')
display(result['uncertainty_summary'])
print('External prediction confidence:')
display(result['prediction_confidence'])
print('Per-member deviation summary:')
display(result['deviation_summary'])

In [ ]:
for figure_name in (
    'gcn3_ensemble_member_metrics.png',
    'gcn3_ensemble_confidence_intervals.png',
    'gcn3_ensemble_uncertainty_vs_error.png',
    'gcn3_ensemble_prediction_deviations.png',
):
    print(figure_name)
    display(Image(filename=str(output_dir / figure_name)))
print('Interpretation report:')
print((output_dir / 'gcn3_ensemble_confidence_report.txt').read_text(encoding='utf-8'))

In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for Colab.')
    git_output_root.mkdir(parents=True, exist_ok=True)
    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODELS_TO_GITHUB:
        for model_path in git_run_dir.rglob('*.pt'):
            model_path.unlink()
    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)
    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'], check=True, capture_output=True, text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)
    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'], check=False)
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            message = f'Add final GCN-3 ensemble confidence results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)
else:
    print('GitHub push disabled.')

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = shutil.make_archive('/content/gcn3_ensemble_uncertainty', 'zip', root_dir=output_dir)
    files.download(archive_path)
else:
    print(f'Outputs and checkpoints are in {output_dir}')